# Financial sentiment regression benchmark (Kaggle)

Self-contained notebook: upload it to Kaggle and attach the benchmark CSV dataset as an **Input**. It locates `fiqa_2018_sentiment_regression.csv` under `/kaggle/input` automatically.

The notebook compares FinBERT, FLANG-RoBERTa, and FLANG-DistilBERT using the same scalar regression head. It preserves FiQA's official train, validation, and test partitions. Early stopping selects the checkpoint by validation RMSE; the test partition is evaluated only after selection.

Training uses AdamW with a lower encoder learning rate than head learning rate, warmup/decay, gradient clipping, mixed precision on CUDA, dropout, and Smooth L1 loss to reduce sensitivity to outlier scores. These are robust starting settings, not a guarantee of a global optimum. Internet must be enabled in Kaggle for the first checkpoint download.

## Dependencies

Run the next cell first. Kaggle generally provides a CUDA-enabled PyTorch installation, which this notebook keeps. If package installation updates an already-imported library, restart the Kaggle session and run from the top.

In [ ]:
%pip install -q "transformers<5" sentencepiece numpy pandas scikit-learn matplotlib

In [ ]:
from pathlib import Path
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore", category=FutureWarning)
INPUT_ROOT = Path("/kaggle/input")
matches = list(INPUT_ROOT.rglob("fiqa_2018_sentiment_regression.csv")) if INPUT_ROOT.exists() else []
if not matches:
    raise FileNotFoundError("Could not find fiqa_2018_sentiment_regression.csv under /kaggle/input. Attach the benchmark dataset as a Kaggle Input.")
FIQA_PATH = matches[0]
OUTPUT_DIR = Path("/kaggle/working/financial_sentiment_regression")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CHECKPOINTS = {
    "FinBERT": "yiyanghkust/finbert-pretrain",
    "FLANG-RoBERTa": "SALT-NLP/FLANG-Roberta",
    "FLANG-DistilBERT": "SALT-NLP/FLANG-DistilBERT",
}
SEED = 42
MAX_LENGTH = 256
BATCH_SIZE = 16  # Lower to 8 or 4 if your Kaggle GPU runs out of memory.
EPOCHS = 10
ENCODER_LR = 1e-5
HEAD_LR = 5e-5
WEIGHT_DECAY = 0.02
WARMUP_RATIO = 0.10
DROPOUT = 0.20
HEAD_HIDDEN_SIZES = (256,)
PATIENCE = 3
HUBER_BETA = 0.10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
SAVE_CHECKPOINTS = False
print("Device:", DEVICE)
print("FiQA:", FIQA_PATH)
print("Model checkpoints:", MODEL_CHECKPOINTS)
print("FiQA is small; use validation metrics for selection and interpret test metrics cautiously.")

In [ ]:
import torch
from torch import nn
from transformers import AutoModel

class RegressionHead(nn.Module):
    def __init__(self, hidden_size, dropout=0.2, hidden_sizes=(256,)):
        super().__init__()
        layers, input_size = [], hidden_size
        for width in hidden_sizes:
            layers.extend([nn.Linear(input_size, width), nn.LayerNorm(width), nn.GELU(), nn.Dropout(dropout)])
            input_size = width
        layers.append(nn.Linear(input_size, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, pooled):
        return self.network(pooled).squeeze(-1)

class FinancialTransformerBase(nn.Module):
    """Shared encoder composition; subclasses attach a task-specific head."""
    def __init__(self, checkpoint, task_head, **model_kwargs):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(checkpoint, **model_kwargs)
        config = self.encoder.config
        hidden_size = getattr(config, "hidden_size", None) or getattr(config, "dim", None) or getattr(config, "d_model", None)
        if hidden_size is None:
            raise ValueError(f"Could not find encoder hidden size for {checkpoint}")
        self.head = task_head(int(hidden_size))

    def pooled_output(self, input_ids, attention_mask, token_type_ids=None):
        inputs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            inputs["token_type_ids"] = token_type_ids
        output = self.encoder(**inputs)
        pooled = getattr(output, "pooler_output", None)
        return pooled if pooled is not None else output.last_hidden_state[:, 0]

class FinancialTextRegressor(FinancialTransformerBase):
    def __init__(self, checkpoint, dropout=0.2, hidden_sizes=(256,)):
        super().__init__(checkpoint, lambda size: RegressionHead(size, dropout, hidden_sizes))

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        return self.head(self.pooled_output(input_ids, attention_mask, token_type_ids))

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_splits():
    frame = pd.read_csv(FIQA_PATH).dropna(subset=["sentence", "sentiment_score", "split"]).copy()
    frame["sentiment_score"] = pd.to_numeric(frame["sentiment_score"], errors="coerce")
    frame = frame.dropna(subset=["sentiment_score"])
    frame["split"] = frame["split"].str.lower()
    split_frames = {name: frame[frame["split"] == name].reset_index(drop=True)
                    for name in ["train", "validation", "test"]}
    if any(part.empty for part in split_frames.values()):
        raise ValueError("FiQA CSV must contain non-empty train, validation, and test splits.")
    return split_frames


class FiQADataset(Dataset):
    def __init__(self, frame, tokenizer):
        self.texts = frame["sentence"].astype(str).tolist()
        self.targets = frame["sentiment_score"].astype(float).tolist()
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.targets)
    def __getitem__(self, index):
        item = self.tokenizer(self.texts[index], truncation=True, max_length=MAX_LENGTH, padding=False)
        item["labels"] = self.targets[index]
        return item


def collator_for(tokenizer):
    def collate(items):
        labels = torch.tensor([item.pop("labels") for item in items], dtype=torch.float32)
        batch = tokenizer.pad(items, padding=True, return_tensors="pt")
        batch["labels"] = labels
        return batch
    return collate


def make_loader(frame, tokenizer, shuffle=False):
    return DataLoader(FiQADataset(frame, tokenizer), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=0, pin_memory=DEVICE.type == "cuda", collate_fn=collator_for(tokenizer))


def regression_metrics(actual, predicted):
    return {"mae": mean_absolute_error(actual, predicted),
            "rmse": float(np.sqrt(mean_squared_error(actual, predicted))),
            "r2": r2_score(actual, predicted)}


def predict(model, loader):
    model.eval()
    actual, predicted = [], []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels")
            batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                scores = model(**batch)
            actual.extend(labels.tolist())
            predicted.extend(scores.cpu().tolist())
    return np.asarray(actual), np.asarray(predicted)

In [ ]:
def train_one_model(model_label, checkpoint, splits):
    set_seed()
    print(f"Loading tokenizer for {model_label}: {checkpoint}")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=False)
    train_loader = make_loader(splits["train"], tokenizer, shuffle=True)
    val_loader = make_loader(splits["validation"], tokenizer)
    test_loader = make_loader(splits["test"], tokenizer)

    model = FinancialTextRegressor(checkpoint, DROPOUT, HEAD_HIDDEN_SIZES).to(DEVICE)
    head_params = list(model.head.parameters())
    head_ids = {id(param) for param in head_params}
    encoder_params = [param for param in model.parameters() if id(param) not in head_ids]
    optimizer = torch.optim.AdamW([
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": head_params, "lr": HEAD_LR},
    ], weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(train_loader) * EPOCHS)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    loss_fn = torch.nn.SmoothL1Loss(beta=HUBER_BETA)
    best_rmse, best_state, stale, history = float("inf"), None, 0, []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        loss_sum, seen = 0.0, 0
        for batch in train_loader:
            labels = batch.pop("labels").to(DEVICE, non_blocking=True)
            batch = {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}
            optimizer.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                predictions = model(**batch)
                loss = loss_fn(predictions, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            loss_sum += loss.item() * labels.size(0)
            seen += labels.size(0)

        y_val, p_val = predict(model, val_loader)
        val_scores = regression_metrics(y_val, p_val)
        row = {"epoch": epoch, "train_huber_loss": loss_sum / max(seen, 1),
               **{f"val_{key}": value for key, value in val_scores.items()}}
        history.append(row)
        print(f"{model_label} | epoch {epoch}: train_loss={row['train_huber_loss']:.4f}, val_RMSE={val_scores['rmse']:.4f}, val_MAE={val_scores['mae']:.4f}")
        if val_scores["rmse"] < best_rmse:
            best_rmse = val_scores["rmse"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print("Early stopping: validation RMSE did not improve.")
                break

    model.load_state_dict(best_state)
    y_test, p_test = predict(model, test_loader)
    result = {"model": model_label, "checkpoint": checkpoint,
              "train_rows": len(splits["train"]), "validation_rows": len(splits["validation"]),
              "test_rows": len(splits["test"]), **regression_metrics(y_test, p_test)}
    if SAVE_CHECKPOINTS:
        torch.save(best_state, OUTPUT_DIR / f"fiqa_{model_label.lower().replace('-', '_')}.pt")
    del model, optimizer, scheduler
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result, pd.DataFrame(history), y_test, p_test

fiqa_splits = load_splits()
print("Rows by split:", {name: len(frame) for name, frame in fiqa_splits.items()})
results, histories, predictions = [], {}, {}
for model_label, checkpoint in MODEL_CHECKPOINTS.items():
    result, history, y_test, p_test = train_one_model(model_label, checkpoint, fiqa_splits)
    results.append(result)
    histories[model_label] = history
    predictions[model_label] = (y_test, p_test)
results_df = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
results_df

In [ ]:
results_df.to_csv(OUTPUT_DIR / "regression_metrics.csv", index=False)
print("Saved metrics:", OUTPUT_DIR / "regression_metrics.csv")

## Visualizations

The plots show validation learning curves, held-out test errors, predicted-versus-actual scores, and residual distributions.

In [ ]:
# Validation learning curves.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for model_label, history in histories.items():
    axes[0].plot(history["epoch"], history["train_huber_loss"], marker="o", label=model_label)
    axes[1].plot(history["epoch"], history["val_rmse"], marker="o", label=model_label)
axes[0].set(title="Training Smooth L1 loss", xlabel="Epoch", ylabel="Loss")
axes[1].set(title="Validation RMSE", xlabel="Epoch", ylabel="RMSE")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "regression_learning_curves.png", dpi=160, bbox_inches="tight")
plt.show()

# Held-out MAE and RMSE comparison.
fig, ax = plt.subplots(figsize=(9, 5))
results_df.set_index("model")[["mae", "rmse"]].plot(kind="bar", ax=ax, rot=15)
ax.set_title("FiQA held-out test error (lower is better)")
ax.set_ylabel("Error in sentiment-score units")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "regression_model_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

# Test predictions and residuals by model.
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for col, model_label in enumerate(MODEL_CHECKPOINTS):
    actual, predicted = predictions[model_label]
    axes[0, col].scatter(actual, predicted, alpha=0.7)
    low = min(float(actual.min()), float(predicted.min()))
    high = max(float(actual.max()), float(predicted.max()))
    axes[0, col].plot([low, high], [low, high], "r--", linewidth=1)
    axes[0, col].set(title=f"{model_label}: actual vs predicted", xlabel="Actual score", ylabel="Predicted score")
    residuals = predicted - actual
    axes[1, col].hist(residuals, bins=18, alpha=0.8)
    axes[1, col].axvline(0, color="red", linestyle="--", linewidth=1)
    axes[1, col].set(title=f"{model_label}: residuals", xlabel="Prediction − actual", ylabel="Count")
for ax in axes.ravel():
    ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "regression_predictions_and_residuals.png", dpi=160, bbox_inches="tight")
plt.show()
print("Plots saved to:", OUTPUT_DIR)

## Reading the results

Lower test MAE/RMSE is better; higher R² is better. FiQA has a small training set, so scores may vary with the random seed and training environment. Preserve the provided splits and do not choose epochs or hyperparameters based on test metrics.